In [69]:
import pandas as pd
import numpy as np

fact_housing = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\fact_housing_updated.csv')
fact_household = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\fact_household.csv')
dim_lender_terms = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\support\dim_lender_terms.csv')


In [27]:
fact_housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 17911 entries, 0 to 17910
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17911 non-null  str    
 1   title             17911 non-null  str    
 2   price             17911 non-null  float64
 3   pricePerSqm       15605 non-null  float64
 4   floorArea         11383 non-null  float64
 5   lotArea           15446 non-null  float64
 6   propertyCategory  14065 non-null  str    
 7   locationLevel     17911 non-null  str    
 8   locationFk        17870 non-null  float64
dtypes: float64(5), str(4)
memory usage: 1.2 MB


In [21]:
dim_lender_terms.head()

,channel,ltv_pct,interest_rate,dti_cap_pct,max_loan_amount,max_term_years,source_confidence
0,Pag-IBIG,0.9,0.0575,0.35,10000000.0,30,official
1,Bank (general),0.8,0.0700,0.30,NaN,20,estimated


In [22]:
terms = dim_lender_terms.set_index("channel").to_dict(orient="index")
pagibig = terms["Pag-IBIG"]
bank = terms["Bank (general)"]

def monthly_pmt(principal_col, annual_rate, years):
    r = annual_rate / 12
    n = years * 12
    pmt = principal_col * r * (1 + r) ** n / ((1 + r) ** n - 1)
    return pmt.round(2)


In [23]:
fact_household.info()

<class 'pandas.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   geographyFk           116 non-null    int64  
 1   families              116 non-null    int64  
 2   reliability           116 non-null    str    
 3   monthlyIncome         116 non-null    int64  
 4   monthlyExpenses       116 non-null    int64  
 5   netIncome             116 non-null    int64  
 6   capacityToPayPagibig  116 non-null    float64
 7   capacityToPayBank     116 non-null    float64
dtypes: float64(2), int64(5), str(1)
memory usage: 7.4 KB


In [33]:

# 2. Separate housing data by location level
housing_city = fact_housing[fact_housing['locationLevel'] == 'city'].copy()
housing_province = fact_housing[fact_housing['locationLevel'] == 'province'].copy()

# 3. Build City Level Table
city_level = pd.merge(
    fact_household,
    housing_city,
    left_on='geographyFk',
    right_on='locationFk',
    how='inner'  
)
# Update geographyFk with locationFk value and remove duplicate key column
city_level['geographyFk'] = city_level['locationFk']
city_level = city_level.drop(columns=['locationFk'])

# 4. Build Province Level Table
province_level = pd.merge(
    fact_household,
    housing_province,
    left_on='geographyFk',
    right_on='locationFk',
    how='inner'  
)
# Update geographyFk with locationFk value and remove duplicate key column
province_level['geographyFk'] = province_level['locationFk']
province_level = province_level.drop(columns=['locationFk'])

# 5. Inspect results
print("City Level Shape:", city_level.shape)
print("Province Level Shape:", province_level.shape)


fact_housing_merge = pd.concat([city_level, province_level], ignore_index=True).drop_duplicates()
print("Number of rows in merged housing: ", fact_housing_merge.shape)
# housing_affordability = pd.merge(city_level,province_level, how='outer',)

City Level Shape: (7761, 16)
Province Level Shape: (3153, 16)
Number of rows in merged housing:  (8378, 16)


In [41]:
# Aggregate the date by geographyFk and location level
grouped_housing = fact_housing_merge.groupby('geographyFk', as_index=False).agg(
    totalProperties=('id','count'),
    avgPrice=('price', 'mean'),
    medianPrice=('price', 'median'),
    avgPricePerSqm=('pricePerSqm', 'mean'),
    medianPricePerSqm=('pricePerSqm', 'median'),
    medianLotArea=('lotArea', 'median'),
    avgLotArea=('lotArea', 'mean')
)

In [42]:
grouped_housing.head(10)

,geographyFk,totalProperties,avgPrice,medianPrice,avgPricePerSqm,medianPricePerSqm,medianLotArea,avgLotArea
0,12900000.0,1,2.839600e+06,2839600.0,23468.000000,23468.0,121.0,121.000000
1,15500000.0,160,1.344661e+06,49000.0,4144.206250,907.0,54.0,1255.412500
2,21500000.0,10,7.734290e+06,3926950.0,8687.444444,1168.0,665.0,5552.700000
3,30800000.0,194,1.582619e+06,1205050.0,25983.768519,27690.0,65.0,174.324074
4,31400000.0,624,2.635661e+06,2155250.0,111543.751645,23140.5,150.5,586.441417
5,34900000.0,32,1.447780e+06,612000.0,4225.750000,1810.5,375.0,452.338125
6,35400000.0,319,1.862155e+06,1489500.0,35850.529781,37623.0,42.0,106.427116
7,35401000.0,76,2.327849e+06,1755900.0,43600.184211,50169.0,35.0,104.098684
8,36916000.0,322,1.313812e+06,1058000.0,23829.978193,26806.0,49.5,109.690031
9,37100000.0,178,1.309626e+06,1179100.0,23264.792135,21590.0,55.0,65.146067


In [ ]:
# df = grouped_housing.copy()
df = pd.merge(fact_household, grouped_housing ,on='geographyFk', how='inner')


df.info()



<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   geographyFk           68 non-null     int64  
 1   families              68 non-null     int64  
 2   reliability           68 non-null     str    
 3   monthlyIncome         68 non-null     int64  
 4   monthlyExpenses       68 non-null     int64  
 5   netIncome             68 non-null     int64  
 6   capacityToPayPagibig  68 non-null     float64
 7   capacityToPayBank     68 non-null     float64
 8   totalProperties       68 non-null     int64  
 9   avgPrice              68 non-null     float64
 10  medianPrice           68 non-null     float64
 11  avgPricePerSqm        67 non-null     float64
 12  medianPricePerSqm     67 non-null     float64
 13  medianLotArea         67 non-null     float64
 14  avgLotArea            67 non-null     float64
dtypes: float64(8), int64(6), str(1)
memo

In [77]:
# city_level = fact_household.merge(fact_housing, on='geographyFk', how='inner')

# df["houseMedianPrice"] = df.groupby("geographyFk")["price"].transform("median")
# df["listingCount"] = df.groupby("geographyFk")["price"].transform("count")
# pagibig details
df["pagIbigRequiredLoan"] = df["medianPrice"] * pagibig["ltv_pct"]
df["pagIbigAmortization"] = monthly_pmt(
    df["pagIbigRequiredLoan"], pagibig["interest_rate"], pagibig["max_term_years"]
)
df["pagIbigCapacity"] = (df["monthlyIncome"] * pagibig["dti_cap_pct"]).round(2)
df["pagIbigGap"] = (df["pagIbigCapacity"] - df["pagIbigAmortization"]).round(2)
df["pagIbigLoanStatus"] = np.where(df["pagIbigGap"] <= 0, "Not Applicable", "Applicable")


# bank details
df["bankRequiredLoan"] = df["medianPrice"] * bank["ltv_pct"]
df["bankAmortization"] = monthly_pmt(
    df["bankRequiredLoan"], bank["interest_rate"], bank["max_term_years"]
)
df["bankCapacity"] = (df["monthlyIncome"] * bank["dti_cap_pct"]).round(2)
df["bankGap"] = (df["bankCapacity"] - df["bankAmortization"]).round(2)
df["bankLoanStatus"] = np.where(df["bankGap"] <= 0, "Not Applicable", "Applicable")


df = df.drop(columns=['monthlyIncome','monthlyExpenses','netIncome','capacityToPayPagibig','capacityToPayBank'])
df.head()



KeyError: 'monthlyIncome'

In [83]:
notApplicable_df = df[df['pagIbigLoanStatus'] == 'Not Applicable']
print('Not applicable: ',notApplicable_df.shape)

applicable_df = df[df['pagIbigLoanStatus'] == 'Applicable']
print('Applicable: ',applicable_df.shape)

Not applicable:  (39, 20)
Applicable:  (29, 20)


In [87]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   geographyFk          68 non-null     int64  
 1   families             68 non-null     int64  
 2   reliability          68 non-null     str    
 3   totalProperties      68 non-null     int64  
 4   medianPrice          68 non-null     float64
 5   medianPricePerSqm    67 non-null     float64
 6   medianLotArea        67 non-null     float64
 7   pagIbigRequiredLoan  68 non-null     float64
 8   pagIbigAmortization  68 non-null     float64
 9   pagIbigCapacity      68 non-null     float64
 10  pagIbigGap           68 non-null     float64
 11  pagIbigLoanStatus    68 non-null     str    
 12  bankRequiredLoan     68 non-null     float64
 13  bankAmortization     68 non-null     float64
 14  bankCapacity         68 non-null     float64
 15  bankGap              68 non-null     float64
 16  ban

In [88]:

# to_drop_columns = ['avgPrice','avgPricePerSqm','avgLotArea']
# df =df.drop(columns=to_drop_columns)
df.info()

saved_location = r'D:\azure_housing_analytics\dataset\gold\fact_affordability_new.csv'
df.to_csv(saved_location, index=False)
print(f"fact_affordability saved to {saved_location}")

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   geographyFk          68 non-null     int64  
 1   families             68 non-null     int64  
 2   reliability          68 non-null     str    
 3   totalProperties      68 non-null     int64  
 4   medianPrice          68 non-null     float64
 5   medianPricePerSqm    67 non-null     float64
 6   medianLotArea        67 non-null     float64
 7   pagIbigRequiredLoan  68 non-null     float64
 8   pagIbigAmortization  68 non-null     float64
 9   pagIbigCapacity      68 non-null     float64
 10  pagIbigGap           68 non-null     float64
 11  pagIbigLoanStatus    68 non-null     str    
 12  bankRequiredLoan     68 non-null     float64
 13  bankAmortization     68 non-null     float64
 14  bankCapacity         68 non-null     float64
 15  bankGap              68 non-null     float64
 16  ban

In [ ]:


to_drop_columns = ['id', 'sourceSlug', 'sourceName', 'title', 'price', 'priceFormatted', 'pricePerSqm', 'floorArea', 'lotArea', 'city', 'province', 'isNew', 'daysListed', 'listingScore', 'firstSeenAt', 'families', 'reliability', 'monthlyIncome', 'monthlyExpenses', 'netIncome']

fact_affordability = df.drop(columns=to_drop_columns)
print(fact_affordability.columns.tolist())

# fact_affordability.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_amortize.csv')
print('table saved')

In [ ]:
total_listings = len(dim_housing)
matched_listings = len(df)

print(f"Total listings in dim_housing: {total_listings}")
print(f"Listings matched to fact_household: {matched_listings}")
print(f"Unmatched (no household income data for their geography): {total_listings - matched_listings}")
print(f"Coverage: {matched_listings / total_listings:.1%}")

In [ ]:
fact_affordability.listingCount